# Test SkipEvent and FinalizeEvent

Test that SkipEvent drops events and FinalizeEvent sends events to sink immediately.

In [ ]:
from bspump.jupyter import *
from bspump.test import TestSink, TestSource

In [ ]:
event = {}
test_events = [
    # Test SkipEvent - event with "skip": True should be dropped (empty output)
    ({"action": "skip", "value": 1}, {"expect": []}),
    # Test FinalizeEvent - event with "finalize": True should go to sink immediately without further processing
    ({"action": "finalize", "value": 2}, {"expect": [{"action": "finalize", "value": 2, "finalized": True}]}),
    # Test normal event - should go through all processing steps
    ({"action": "normal", "value": 3}, {"expect": [{"action": "normal", "value": 3, "step1": True, "step2": True}]}),
]

In [ ]:
auto_pipeline(
    source=lambda app, pipeline: TestSource(app, pipeline, "TestSource"),
    sink=lambda app, pipeline: TestSink(app, pipeline, "TestSink")
)

In [ ]:
# Step 1: Handle SkipEvent and FinalizeEvent
if event.get("action") == "skip":
    print(f"Skipping event with value {event['value']}")
    raise SkipEvent()

if event.get("action") == "finalize":
    print(f"Finalizing event with value {event['value']}")
    event["finalized"] = True
    raise FinalizeEvent(event)

# Normal processing
event["step1"] = True
print(f"Step 1 complete for event {event}")

In [ ]:
# Step 2: This should only run for normal events (not skipped or finalized)
event["step2"] = True
print(f"Step 2 complete for event {event}")